# Souty gesture recognizer retraining — Tunisian Sign Language + stock gestures merged

Retrains MediaPipe's gesture recognizer on **23 pruned, single-hand signs** from the **[Tunisian Sign Language Dataset](https://www.kaggle.com/datasets/warcoder/tunisian-sign-language-dataset)** (Chakroun & Jerbi, 2023 — Mendeley Data, collected with ATILS, the Tunisian Association of Sign Language Interpreters) **merged with the original 10-class HaGRID vocabulary** (stock-gesture equivalents + `OK_Sign`/`Three_Fingers`/`Four_Fingers`/`Shaka_Sign`) and HaGRID's `no_gesture` folder as the required negative ("none") class. **34 classes total.**

**Why pruned, and why single-hand (TunSL side):** an earlier run trained all 57 TunSL signs (test accuracy 60.22%). Diagnosis afterward found the real cause: MediaPipe Model Maker's dataset loader hard-codes `num_hands=1` and only keeps one hand's landmarks per image — confirmed directly in its [source](https://github.com/google-ai-edge/mediapipe/blob/master/mediapipe/model_maker/python/vision/gesture_recognizer/dataset.py). The live app classifies each detected hand independently too, with no joint two-hand concept. Running the real MediaPipe hand detector over all 4423 training photos found **28 of the 57 signs (49%) are majority two-handed** in their own training photos — those can never be learned correctly by this architecture, no matter the data or epochs. `training/README.md` has the full per-sign hand-count audit and the excluded-signs table. This run keeps only the **23 signs** that are reliably single-hand (≥75% single-hand purity among detected hands, ≥15 clean examples).

**Why the old HaGRID vocabulary is back:** the original 10-class run (commit `f1ca435`) already proved out mapping HaGRID's own gesture folders — which live under the *same* attached dataset as the "none" source, despite its `-no-gesture` slug — onto the stock-7-equivalent names plus 4 extras, reaching 95.2% accuracy on that problem alone. This run reuses that exact mapping unchanged. `ILoveYou` stays excluded: HaGRID has no matching gesture, same as the original run — bringing it back for real needs your own photos in a new Kaggle dataset, not just a dictionary entry.

**Also fixed: the train/test split itself.** The TunSL images are video key frames (confirmed in its bundled report), and measuring perceptual-hash similarity confirms consecutively-numbered files are near-duplicate frames. Model Maker's default `Dataset.split()` shuffles randomly before slicing, which would scatter near-duplicate frames across train and test and inflate the printed accuracy. This run splits each class's own capture-ordered files into contiguous train/validation/test blocks instead — see step 4's cell for the full reasoning.

**Setup before running:**
1. Settings -> Accelerator -> GPU.
2. Add Data -> attach **both**: `warcoder/tunisian-sign-language-dataset` and `innominate817/hagrid-classification-512p-no-gesture` (this one dataset supplies the HaGRID classes, `none`, *and* the old stock-gesture equivalents — nothing else to attach).
3. Run all cells top to bottom.
4. **Check step 4's printed output before trusting the training run** — it prints exactly which TunSL/HaGRID folders were kept vs. skipped, and each class's train/val/test split sizes; any `missing` warning means a `_ROOT` path needs adjusting against the Input panel first.
5. When done, download `exported_model/gesture_recognizer.task` from the Output panel (top right).
6. Locally: copy that file to `../public/models/gesture_recognizer.task` — `lib/gestureDictionary.ts` already has the matching combined 34-class mapping.

**This replaces the vocabulary of any previously deployed model** (the stock-7-only model, the 10-class HaGRID model, or the 57-sign TunSL-only run) with this merged 34-class one.

**No accuracy guarantee:** pruning the TunSL side removes the architectural mismatch and the ordered split removes most of the near-duplicate-frame leakage, but TunSL per-class image counts are still modest (16–35 for the thinnest kept classes, meaning test blocks as small as 2-3 images) and merging 34 classes together is untested as one combined problem — some TunSL/HaGRID pairs could be more confusable together than either group was alone. Manually test all 34 signs after deploying — aggregate test accuracy hides individual bad classes.

**Why Python 3.10, not the notebook's own kernel:** `mediapipe-model-maker` depends on `tf-models-official==2.11.6`, which pins `pyyaml<6.0` - a version with no prebuilt wheel for Python 3.12 (Kaggle's current default). Building it from source currently fails there - a live, unresolved upstream bug ([mediapipe-samples#643](https://github.com/google-ai-edge/mediapipe-samples/issues/643)). Installing Python 3.10 alongside the default and running under that is the confirmed community workaround, so training runs as a subprocess under python3.10 rather than in the notebook's own cells.

## 1. Install Python 3.10 and mediapipe-model-maker under it

In [ ]:
!sudo apt-get install -y python3.10 python3.10-distutils python3.10-venv
!wget -q https://bootstrap.pypa.io/get-pip.py -O /tmp/get-pip.py
!python3.10 /tmp/get-pip.py
!python3.10 -m pip install mediapipe-model-maker

## 2. Force a GPU-enabled TensorFlow install

`mediapipe-model-maker` pulls in plain `tensorflow`, which needs CUDA/cuDNN already present and version-matched on the system - unreliable for a manually-installed python3.10 that isn't Kaggle's own preconfigured environment. `tensorflow[and-cuda]` bundles the matching NVIDIA libraries as pip packages instead, so it works regardless of what's on the host. Pinned to `2.15.0.post1` specifically: plain `2.15.1` (what step 1 installed) has a known broken `tensorrt` dependency in the `[and-cuda]` extra that `.post1` fixed.

In [ ]:
!python3.10 -m pip install "tensorflow[and-cuda]==2.15.0.post1"

## 3. Verify GPU is actually visible under python3.10

If this still prints an empty list, training will fall back to CPU (much slower, but not broken) - check Settings -> Accelerator is set to GPU and re-run from step 1 if you changed it after starting.

In [ ]:
!python3.10 -c "import tensorflow as tf; print('GPUs:', tf.config.list_physical_devices('GPU'))"

## 4. Write the training script

`find_leaf_class_dirs()` recursively discovers every folder that directly contains images and nothing else (no subfolders) - handles both Kaggle's occasional extra duplicate-name mount nesting *and* this dataset's own category/sign nesting (e.g. `Famille/<sign name>/`), without needing to know the exact structure in advance. `KEPT_SIGN_LABELS` is the pruned, audited 23-sign TunSL list (see the intro cell above and `training/README.md`) - any discovered TunSL folder not in that set is intentionally skipped and printed as such, not an error. `HAGRID_LABEL_MAP` adds the 10 old stock-gesture-equivalent classes back in, sourced from the same attached HaGRID dataset as `none`.

**Split:** each class's files are split into train/validation/test as contiguous *capture-order* blocks (`block_split()`), not a random shuffle - the TunSL images are video key frames, and a random split would scatter near-duplicate adjacent frames across train and test (see the training script's own docstring for the measured evidence). Three separate dataset folders are built and loaded as three separate `Dataset.from_folder()` calls instead of one `Dataset.split()`. Also prints `Dataset.size` for each split after MediaPipe's own hand-detection filtering runs (confirmed via the official Model Maker source: images with no detected hand are silently dropped) alongside the raw copied-in count, so a big gap is visible before a training run rather than discovered after.

In [ ]:
%%writefile /kaggle/working/_train.py
"""Retrains the gesture recognizer on a PRUNED, single-hand subset of the
real Tunisian Sign Language dataset (warcoder/tunisian-sign-language-dataset
on Kaggle, sourced from Chakroun & Jerbi 2023's Mendeley dataset, collected
with ATILS - the Tunisian Association of Sign Language Interpreters),
MERGED with the original 10-class HaGRID vocabulary (stock-gesture
equivalents + OK_Sign/Three_Fingers/Four_Fingers/Shaka_Sign) and HaGRID's
no_gesture folder as the required negative ("none") class.

Must be run under Python 3.10, not whatever Python a Kaggle/Colab
notebook's own kernel defaults to (currently 3.12). mediapipe-model-maker
pulls in tf-models-official==2.11.6, which pins pyyaml<6.0 - a version with
no prebuilt wheel for 3.12, so pip tries to build it from source and that
build is currently broken (confirmed, unresolved upstream as of early
2026: https://github.com/google-ai-edge/mediapipe-samples/issues/643).
Installing Python 3.10 alongside the default and running under that is the
confirmed community workaround.

kaggle_pipeline.ipynb writes this file to /kaggle/working/_train.py and
runs it with `MPLBACKEND=Agg python3.10 /kaggle/working/_train.py` - see
that notebook for the full setup (installing python3.10, forcing a
GPU-enabled tensorflow[and-cuda] install, attaching both Kaggle datasets,
etc). Not meant to be run directly under a notebook's own kernel cells.

MPLBACKEND=Agg matters if you invoke this manually too: a notebook
kernel's own MPLBACKEND env var (set for inline plotting) leaks into any
subprocess and crashes matplotlib, which mediapipe pulls in transitively
and which isn't actually used for output here.

WHY ONLY 23 OF THE 57 TUNSL SIGNS, NOT ALL OF THEM:
The first full-57-class run (test accuracy 60.22%) turned out to be
measuring the wrong thing - confirmed by reading Model Maker's own source
(mediapipe/model_maker/python/vision/gesture_recognizer/dataset.py):
its internal HandLandmarker is hard-coded to `num_hands=1` and only ever
keeps `hand_landmarks[0]`, so ANY image with two hands silently has one
of them discarded during training - and the live app
(components/GestureScanner.tsx) classifies each detected hand
independently at inference time too. Neither side has any concept of
"these two hands together are one sign" - this is a hard capability
ceiling of the architecture, not a tuning problem.

To find out how much of the vocabulary that actually breaks, every one of
the dataset's 4423 training photos was run through the real MediaPipe
HandLandmarker (2-hand mode) to count how many hands each photo actually
shows. Result: 28 of the 57 signs (49%) are majority two-handed in their
own training photos (some, like "cv", "demande", "om", "jom3a", are 100%
two-handed) - those can never be learned correctly by this model no
matter how much data or how many epochs are thrown at them, since
training only ever sees one arbitrary hand of the pair, inconsistently,
image to image. KEPT_SIGN_LABELS below is the subset that survived:
>=75% single-hand purity among images where a hand was detected at all,
and >=15 clean single-hand training images after that filter (two
exceptions made by judgment, not the raw numbers: "non" kept at 74%
purity with 48 clean images, for basic yes/no symmetry with "oui"; and
the dropped-for-thinness list includes "mar7ba" and "karhba" even though
one of them hit 100% purity, because 9-13 images is too few to trust
regardless of how clean they are). See training/README.md for the full
excluded-signs table before adding any TunSL sign back.

WHY THE OLD HaGRID VOCABULARY IS MERGED BACK IN:
An earlier run (commit f1ca435, "Expand gesture vocabulary to 10 classes")
already proved out mapping HaGRID's own gesture folders - which live
under the SAME attached dataset as the "none" source, despite its
`-no-gesture` slug - onto the original stock-7-equivalent names plus 4
extras, reaching 95.2% accuracy on that 10-class problem alone.
HAGRID_LABEL_MAP below reuses that exact, already-proven mapping rather
than guessing a new one. ILoveYou is intentionally still excluded, same
as that original run: HaGRID has no matching gesture (thumb+index+pinky
extended), so it was dropped there too - bringing it back for real would
need ~100 of your own photos of that hand shape in a new Kaggle dataset,
not just a dictionary entry.

WHY A CUSTOM ORDERED SPLIT, NOT Dataset.split():
The TunSL dataset's own bundled report ("Mobile Application for Tunisian
Sign Language (TunSL) Recognition.pdf") says its images are OpenCV key
frames extracted from videos of signers, not independent photographs:
"We extracted frames from videos containing signs using OpenCV. We
acquired sets of frames that have a good representation of the shots...
while eliminating most of the repeated frames." Measuring this directly
(phash perceptual hashing across several classes) confirms it:
consecutively numbered files within a class average a Hamming distance
of 1-7 (near-duplicate frames), versus 7-28 for files far apart in the
same class - filename order preserves video/capture order.

Model Maker's own `Dataset.from_folder()` shuffles the full example list
with an unseeded `random.shuffle()` before `Dataset.split()` takes a
contiguous slice of that shuffled order (both confirmed directly in the
official source: dataset.py and core/data/dataset.py). Combined with the
near-duplicate-frame finding above, that means a random per-image split
will very likely place near-identical frames from the same capture
session in both the train and test sets - the model can partly "pass"
the test by recognizing a frame it has essentially already seen, rather
than by generalizing the hand shape to a new person's hand. The printed
test accuracy from a shuffle-and-slice split should NOT be trusted as a
real estimate of how this model performs on an unseen hand (e.g. the
live app's actual user) - which is exactly the number that matters here.
(HaGRID's own classes are independent crowd-sourced photos, not video
frames, so this risk is specific to the TunSL portion - but the same
ordered-block split is applied uniformly to every class for one simple,
consistent pipeline; it does no harm there.)

Fix: split each class's OWN sorted (capture-order) file list into
contiguous train/validation/test blocks BEFORE calling Model Maker at
all, then load three separate Dataset objects (one per split directory)
instead of one Dataset.split() in two. This isn't a perfect signer-level
holdout for TunSL (the dataset's filenames don't carry a documented
signer ID, only a capture-order sequence number), but holding out the
LAST ~10% of each class's capture-ordered frames for test is a
meaningfully stronger estimate of generalization than randomly
interleaving near-duplicate frames across all three splits.

DATASET FORMAT: verified against the official Model Maker docs
(https://developers.google.com/edge/mediapipe/solutions/customization/gesture_recognizer)
- required layout is flat, <dataset_path>/<label_name>/<img_name>.*, with
exactly one label folder literally named "none" ("the none label
represents any gesture that isn't classified as one of the other
gestures"). The docs also confirm Model Maker runs its own hand detector
during loading and silently drops any image with no hand found
(Dataset.size, in the official source, is set to the count that survives
this, not the count you handed it) - train_and_export() below prints
that gap explicitly for each split so a silent loss isn't discovered
only after a training run.

VOCABULARY SUMMARY: this run trains 23 TunSL signs + 10 HaGRID-derived
classes (Open_Palm, Closed_Fist, Thumb_Up, Thumb_Down, Victory,
Pointing_Up, OK_Sign, Three_Fingers, Four_Fingers, Shaka_Sign) + none = 34
classes total. lib/gestureDictionary.ts already has the matching combined
mapping.

NO ACCURACY GUARANTEE: pruning the TunSL side to single-hand-majority,
high-purity signs removes the architectural mismatch, and the ordered
split removes most of the near-duplicate-frame leakage, but TunSL class
counts are still small (as few as 16-35 images for some, meaning test
blocks as small as 2-3 images for the leanest classes - noisy per-class
signal) and photo conditions vary. The HaGRID-derived classes are much
better-resourced (capped at HAGRID_CLASS_CAP each) and already proved
reliable once at 95.2% on their own, but merging 34 classes together is
untested as a single combined problem - some TunSL/HaGRID pairs could
turn out more confusable together than either group was alone (e.g.
Pointing_Up vs assam, both single raised fingers near the body). Watch
the printed test accuracy, and more importantly, manually test every one
of these 34 signs individually after deploying (aggregate accuracy hides
individual bad classes) - some may still need to be dropped.
"""

import os
import shutil
import unicodedata
from pathlib import Path

from mediapipe_model_maker import gesture_recognizer

# Adjust these against the Input panel's file browser after attaching -
# best guesses based on Kaggle's usual /kaggle/input/datasets/<owner>/<slug>
# mount pattern, same as what we had to confirm for HaGRID last time.
TUNSL_ROOT = Path("/kaggle/input/datasets/warcoder/tunisian-sign-language-dataset")
# Despite the "-no-gesture" slug, this dataset also holds HaGRID's regular
# gesture folders (palm, fist, like, dislike, peace, one, ok, three, four,
# call) alongside no_gesture - confirmed in the original 10-class run
# (commit f1ca435) and reused as-is here, same root for both purposes.
HAGRID_ROOT = Path(
    "/kaggle/input/datasets/innominate817/hagrid-classification-512p-no-gesture"
)

TRAIN_DIR = Path("/kaggle/working/dataset_train")
VAL_DIR = Path("/kaggle/working/dataset_val")
TEST_DIR = Path("/kaggle/working/dataset_test")
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
VAL_FRACTION = 0.1
TEST_FRACTION = 0.1
# Keeps "none" roughly in scale with the real sign classes rather than
# letting HaGRID's much larger pool dominate training and bias the model
# toward predicting "none".
NONE_CAP = 80
# HaGRID has 10k+ images per class - far more than the TunSL classes have,
# so cap each HaGRID-derived class to keep the combined problem reasonably
# balanced and the Kaggle session time reasonable (hand-detection
# preprocessing runs per image).
HAGRID_CLASS_CAP = 120

# The 23 TunSL signs that survived the single-hand purity/sample-size
# audit (see this module's docstring and training/README.md for the full
# excluded-signs table). Every other TunSL folder discovered on disk is
# skipped intentionally, not an error - this is NOT the full dataset's
# class list.
KEPT_SIGN_LABELS = {
    # Demandes (7)
    "assam", "labes", "oui", "non", "siye7a", "ta3raf", "5adamet",
    # Destinations (2)
    "mostawsaf", "sbitar",
    # Famille (7)
    "bent", "bou", "eben", "jad", "jadda", "mar2a", "tfol",
    # Jours (3)
    "5mis", "sebt", "thnin",
    # Transport (4)
    "car", "louage", "metro", "train",
}

# HaGRID source folder name -> our GESTURE_WORD_MAP label (must match
# lib/gestureDictionary.ts exactly). Reused unchanged from the original
# proven 10-class run (commit f1ca435) - no HaGRID equivalent for
# ILoveYou, so it stays excluded (see module docstring).
HAGRID_LABEL_MAP = {
    "palm": "Open_Palm",
    "fist": "Closed_Fist",
    "like": "Thumb_Up",
    "dislike": "Thumb_Down",
    "peace": "Victory",
    "one": "Pointing_Up",
    "ok": "OK_Sign",
    "three": "Three_Fingers",
    "four": "Four_Fingers",
    "call": "Shaka_Sign",
}


def sanitize_label(name: str) -> str:
    """ASCII-folds accented characters (e.g. "metro"'s e) so a label can't
    carry an encoding glitch through Kaggle's mount, this repo, and JS
    string handling later - a mangled accented character was seen when
    inspecting a local copy of this dataset on Windows."""
    normalized = unicodedata.normalize("NFKD", name)
    ascii_only = normalized.encode("ascii", "ignore").decode("ascii")
    return ascii_only if ascii_only else name


def find_leaf_class_dirs(root: Path) -> list[Path]:
    """Every directory that directly contains image files AND has no
    subdirectories - each becomes one training class. A directory with
    both images and subfolders is skipped (likely a stray thumbnail
    sitting alongside real category/sign subfolders, not a class itself)."""
    leaves = []
    for dirpath, dirnames, filenames in os.walk(root):
        has_images = any(Path(f).suffix.lower() in IMAGE_EXTS for f in filenames)
        if has_images and not dirnames:
            leaves.append(Path(dirpath))
    return leaves


def block_split(files: list[Path]) -> tuple[list[Path], list[Path], list[Path]]:
    """Splits an already capture-ordered file list into contiguous
    train/validation/test blocks (no shuffling). Keeping each block
    contiguous, rather than randomly interleaving, avoids putting
    near-duplicate adjacent video frames on both sides of the split - see
    this module's docstring. At least 1 file is guaranteed in validation
    and test whenever the class has >=3 files."""
    n = len(files)
    n_test = max(1, round(n * TEST_FRACTION)) if n >= 3 else 0
    n_val = max(1, round(n * VAL_FRACTION)) if n >= 3 else 0
    n_train = n - n_val - n_test
    if n_train < 1:
        n_train = n
        n_val = n_test = 0
    return files[:n_train], files[n_train:n_train + n_val], files[n_train + n_val:]


def copy_class(src: Path, label: str, cap: int | None = None) -> dict[str, int]:
    """Copies one class's images into TRAIN_DIR/VAL_DIR/TEST_DIR using a
    capture-order block split (see block_split). Returns the per-split
    counts actually copied."""
    files = sorted(f for f in src.iterdir() if f.suffix.lower() in IMAGE_EXTS)
    if cap is not None:
        files = files[:cap]
    train_files, val_files, test_files = block_split(files)

    counts = {}
    for split_dir, split_files, split_name in (
        (TRAIN_DIR, train_files, "train"),
        (VAL_DIR, val_files, "validation"),
        (TEST_DIR, test_files, "test"),
    ):
        dst = split_dir / label
        dst.mkdir(parents=True, exist_ok=True)
        for f in split_files:
            shutil.copy(f, dst / f.name)
        counts[split_name] = len(split_files)
    return counts


def build_dataset() -> dict[str, int]:
    """Returns the total images copied per split, so train_and_export()
    can compare each against that split's Dataset.size (the count that
    actually survives Model Maker's own hand-detection filtering - see
    the module docstring's note on HAND-DETECTION FILTERING)."""
    for d in (TRAIN_DIR, VAL_DIR, TEST_DIR):
        d.mkdir(parents=True, exist_ok=True)

    totals = {"train": 0, "validation": 0, "test": 0}

    # --- TunSL signs ---
    all_sign_dirs = find_leaf_class_dirs(TUNSL_ROOT)
    print(f"Discovered {len(all_sign_dirs)} TunSL sign folders under {TUNSL_ROOT}")

    found_labels = set()
    skipped = []
    for d in sorted(all_sign_dirs):
        label = sanitize_label(d.name)
        if label not in KEPT_SIGN_LABELS:
            skipped.append(label)
            continue
        found_labels.add(label)
        counts = copy_class(d, label)
        for k in totals:
            totals[k] += counts[k]
        print(f"  KEEP [TunSL]   {label}: train={counts['train']} val={counts['validation']} test={counts['test']}")
    print(f"TunSL images copied across {len(found_labels)} kept classes")
    print(f"Skipped {len(skipped)} TunSL folders not in KEPT_SIGN_LABELS (two-handed-majority or too thin): {sorted(skipped)}")

    missing = KEPT_SIGN_LABELS - found_labels
    if missing:
        print(
            "WARNING: some KEPT_SIGN_LABELS were not found on disk - "
            f"check TUNSL_ROOT against the Input panel: {sorted(missing)}"
        )

    # --- HaGRID-derived classes (old stock-gesture equivalents + extras) ---
    hagrid_dirs = find_leaf_class_dirs(HAGRID_ROOT)
    hagrid_by_name = {d.name: d for d in hagrid_dirs}

    found_hagrid = set()
    for src_name, label in HAGRID_LABEL_MAP.items():
        d = hagrid_by_name.get(src_name)
        if d is None:
            print(f"WARNING: no '{src_name}' folder found under {HAGRID_ROOT} - skipping {label}")
            continue
        found_hagrid.add(label)
        counts = copy_class(d, label, cap=HAGRID_CLASS_CAP)
        for k in totals:
            totals[k] += counts[k]
        print(f"  KEEP [HaGRID]  {label}: train={counts['train']} val={counts['validation']} test={counts['test']}")

    missing_hagrid = set(HAGRID_LABEL_MAP.values()) - found_hagrid
    if missing_hagrid:
        print(
            "WARNING: some HaGRID-derived classes were not found on disk - "
            f"check HAGRID_ROOT against the Input panel: {sorted(missing_hagrid)}"
        )

    # --- none (negative class, also from HaGRID) ---
    none_dir = hagrid_by_name.get("no_gesture")
    if none_dir is None:
        raise SystemExit(
            f"no 'no_gesture' folder found under {HAGRID_ROOT} - "
            "cannot train without a none class. Check HAGRID_ROOT."
        )
    none_counts = copy_class(none_dir, "none", cap=NONE_CAP)
    print(f"none: train={none_counts['train']} val={none_counts['validation']} test={none_counts['test']} (from {none_dir})")
    for k in totals:
        totals[k] += none_counts[k]

    print(f"TOTAL images copied across {len(found_labels) + len(found_hagrid) + 1} classes: {totals}")
    return totals


def train_and_export(images_copied: dict[str, int]) -> None:
    hparams_pp = gesture_recognizer.HandDataPreprocessingParams()
    train_data = gesture_recognizer.Dataset.from_folder(dirname=str(TRAIN_DIR), hparams=hparams_pp)
    validation_data = gesture_recognizer.Dataset.from_folder(dirname=str(VAL_DIR), hparams=hparams_pp)
    test_data = gesture_recognizer.Dataset.from_folder(dirname=str(TEST_DIR), hparams=hparams_pp)

    # Model Maker silently drops any image where its own hand detector
    # doesn't find a hand (confirmed in the official dataset.py source -
    # `size=len(valid_hand_data)`). A big gap here means a lot of copied
    # images aren't actually being trained on - worth knowing before
    # trusting the run, especially for the thinner classes.
    for name, data, copied_key in (
        ("train", train_data, "train"),
        ("validation", validation_data, "validation"),
        ("test", test_data, "test"),
    ):
        dropped = images_copied[copied_key] - data.size
        print(f"{name}: Dataset.size={data.size} (copied in: {images_copied[copied_key]}, dropped: {dropped})")
    if test_data.size < 30:
        print(
            "WARNING: test set has very few images - per-class test accuracy "
            "will be noisy (some TunSL classes may have only 1-2 test images). "
            "Manual testing with your own hand matters more than this number."
        )

    # 34 classes total (23 TunSL + 10 HaGRID-derived + none) is a bigger
    # problem than either group alone - keep epochs/dropout from the
    # 23-sign run; the classifier head is still tiny (~2K params) so
    # overfitting risk doesn't scale down just because more classes were
    # added. Watch train vs. validation accuracy in the printed log;
    # validation accuracy plateauing or falling while train accuracy keeps
    # climbing means overfitting, and epochs should come down.
    hparams = gesture_recognizer.HParams(
        export_dir="/kaggle/working/exported_model",
        epochs=30,
    )
    model_options = gesture_recognizer.ModelOptions(dropout_rate=0.2)
    options = gesture_recognizer.GestureRecognizerOptions(
        model_options=model_options, hparams=hparams
    )
    model = gesture_recognizer.GestureRecognizer.create(
        train_data=train_data,
        validation_data=validation_data,
        options=options,
    )

    loss, accuracy = model.evaluate(test_data, batch_size=1)
    print(f"Test loss: {loss:.4f}, test accuracy: {accuracy:.4f}")
    print(
        "This number comes from a capture-order held-out block per TunSL "
        "class (HaGRID classes are independent photos, order doesn't "
        "matter for them) - a more honest estimate than the first run's "
        "60.22%, but still not a substitute for testing every one of these "
        "34 signs by hand once deployed (see ../TESTING.md), especially "
        "the thinnest TunSL classes and any TunSL/HaGRID pair that looks "
        "visually similar (e.g. Pointing_Up vs assam)."
    )

    model.export_model()
    print("Download /kaggle/working/exported_model/gesture_recognizer.task from the Output panel.")


if __name__ == "__main__":
    copied = build_dataset()
    train_and_export(copied)


## 5. Run it

`MPLBACKEND=Agg` overrides the notebook kernel's own `MPLBACKEND` value (set for its inline plotting), which would otherwise leak into this subprocess and crash matplotlib - it's only pulled in transitively by mediapipe's drawing utilities here, not actually used for output, so a plain headless backend is fine.

A `cannot import name 'runtime_version' from 'google.protobuf'` error may print partway through - that's from an optional `tensorflow_datasets` submodule failing to import; it's caught internally and does not stop the script, so it's safe to ignore.

**Stop and check the printed class list (step 4's output) before trusting a full run** - any `missing`/`unexpected` warning means fixing the ROOT paths first, rather than letting a wrong dataset train for 40 epochs.

In [ ]:
!MPLBACKEND=Agg python3.10 /kaggle/working/_train.py